In [ ]:
# EDA and modeling
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib

print('imports OK')


In [ ]:
# Load data
df = pd.read_csv('../data/creditcard.csv')
print('df shape:', df.shape)
print(df['Class'].value_counts())


In [ ]:
# Preprocess (use separate scalers and safe assignment)
scaler_amount = StandardScaler()
df['scaled_amount'] = scaler_amount.fit_transform(df[['Amount']])
scaler_time = StandardScaler()
df['scaled_time'] = scaler_time.fit_transform(df[['Time']])
df.drop(['Time', 'Amount'], axis=1, inplace=True)
print('preprocess done')


In [ ]:
# Balance the data with SMOTE (guard k_neighbors against tiny minority)
X = df.drop('Class', axis=1)
y = df['Class']
print('X shape, y shape:', X.shape, y.shape)
print('class counts:\n', y.value_counts())
min_count = y.value_counts().min()
k = min(5, max(1, min_count - 1))
print('using SMOTE k_neighbors =', k)
smote = SMOTE(k_neighbors=k)
X_res, y_res = smote.fit_resample(X, y)
print('resampled shapes:', X_res.shape, y_res.shape)


In [ ]:
# Split, train, evaluate
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3, random_state=42)
model = RandomForestClassifier(n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
print('\n', classification_report(y_test, model.predict(X_test)))


In [ ]:
# Save model (create models directory if missing)
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/model.pkl')
print('model saved to ../models/model.pkl')
